# LSI Lite – lastoptimiert slicen aus einem FEM-Ergebnis

Aus einer `.vtu`-Datei mit Spannungsergebnis (z. B. FreeCAD/CalculiX) entsteht G-Code für einen 3-Achs-Drucker, dessen Schichten der Zugspannung folgen.

**So geht's:** beide Zellen nacheinander mit ▶ starten. Zelle 2 zeigt die Bedienoberfläche: Bauteil wählen, Einstellungen setzen, *Slicen*.

Läuft in Google Colab und lokal im geklonten Ordner (VS Code, JupyterLab); Einrichtung siehe README. Vereinfachte Fassung des LSI aus der Diplomarbeit von Gereon Zündorf.

In [ ]:
#@title 1  Einrichten
import importlib.util, subprocess, sys, urllib.request
from pathlib import Path
if "google.colab" in sys.modules:
    # Colab: fehlende Pakete und das Programm von GitHub holen
    VERSION = "main"   # fester Stand, z. B. "v1.0"
    BASIS = f"https://raw.githubusercontent.com/gereon1112-web/lsi-lite/{VERSION}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    "vtk==9.7.2", "plotly==7.1.0"], check=True)
    for datei in ["lsi_lite.py", "beispiele/tragwerk.vtu", "beispiele/biegebalken.vtu"]:
        if not Path(datei).exists():
            Path(datei).parent.mkdir(exist_ok=True)
            urllib.request.urlretrieve(f"{BASIS}/{datei}", datei)
else:
    # lokal (geklonter Ordner): fehlende Pakete aus requirements.txt in die gewaehlte Umgebung
    fehlt = [m for m in ("numpy", "scipy", "vtkmodules", "shapely", "contourpy", "matplotlib",
                         "plotly", "ipywidgets", "nbformat") if importlib.util.find_spec(m) is None]
    if fehlt:
        print(f"Es fehlen {', '.join(fehlt)} - wird einmalig installiert (etwa 1 Minute) ...")
        if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                           "-r", "requirements.txt"]).returncode:
            raise SystemExit("Installation fehlgeschlagen. Oben rechts unter 'Kernel auswählen' eine "
                             "eigene Umgebung anlegen (README, 'Notebook in VS Code') und neu starten.")
import lsi_lite
print("Bereit.")

In [ ]:
#@title 2  LSI Lite (Code eingeklappt; zum Ansehen aufklappen)
# Die Bedienoberfläche: Karten mit Einstellungen, Knopf Slicen, Schrittliste und Reiter.
import sys
from pathlib import Path

import ipywidgets as W
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
UPLOAD = "eigene Datei hochladen"
FULL = W.Layout(width="100%")
LABEL = {"description_width": "92px"}

CSS = """<style>
.lsi-card,.lsi-title{font-family:system-ui,-apple-system,Segoe UI,sans-serif}
.lsi-card{flex:1 1 360px;min-width:250px;padding:10px 14px 12px;margin:0 12px 12px 0;
  border:1px solid rgba(128,128,128,.28);border-radius:12px;box-sizing:border-box}
.lsi-card h4{margin:0 0 8px;font-size:13px;letter-spacing:.04em;text-transform:uppercase;opacity:.7}
.lsi-hint{font-size:12px;opacity:.7;line-height:1.4;margin:2px 0 0 96px}
.lsi-title h2{margin:0 0 2px;font-weight:650}
.lsi-title p{margin:0 0 12px;opacity:.75}
</style>"""


def card(title, *children):
    """Ein Kasten mit Ueberschrift; mehrere stehen nebeneinander und brechen auf schmalen Fenstern um."""
    box = W.VBox([W.HTML(f"<h4>{title}</h4>"), *children])
    box.add_class("lsi-card")
    return box


def hint(text):
    """Kleiner erklaerender Text unter einem Bedienelement."""
    return W.HTML(f'<div class="lsi-hint">{text}</div>')


def uploaded_files(upload):
    """Name und Inhalt der hochgeladenen Dateien; ipywidgets 7 (Colab) und 8 liefern sie verschieden."""
    value = upload.value
    if isinstance(value, dict):
        return [(name, bytes(f["content"])) for name, f in value.items()]
    return [(f["name"], bytes(f["content"])) for f in value]


def window_text(nozzle):
    """Schichthoehe und Bahnbreite zur Duese, als Hinweistext."""
    h_min, h_max, _, w = lsi_lite.layer_window(nozzle)
    return (f"Schicht {h_min:.2f} &ndash; {h_max:.2f} mm, Bahn {w:.2f} mm "
            f"({lsi_lite.LAYER_RATIO[0]:.0%} bis {lsi_lite.LAYER_RATIO[1]:.0%} der D&uuml;se)")


def app():
    """Zeigt Eingabe, Einstellungen, Fortschritt und Ergebnis als eine Oberflaeche."""
    s = lsi_lite.SETTINGS
    source = W.Dropdown(options=[f"Beispiel: {n}" for n in lsi_lite.EXAMPLES] + [UPLOAD],
                        description="Bauteil", style=LABEL, layout=FULL)
    upload = W.FileUpload(accept=".vtu", multiple=False, description=".vtu wählen",
                          layout=W.Layout(display="none", margin="4px 0 0 96px"))
    build_dir = W.Dropdown(options=["auto", *lsi_lite.AXES], value=s["build_dir"],
                           description="Baurichtung", style=LABEL, layout=FULL)
    nozzle = W.BoundedFloatText(value=s["nozzle"], min=0.1, max=2.0, step=0.05,
                                description="Düse (mm)", style=LABEL, layout=FULL)
    window = hint(window_text(s["nozzle"]))
    tilt = W.IntSlider(value=s["max_tilt"], min=5, max=45, description="Neigung max",
                       style=LABEL, layout=FULL, readout_format="d")
    infill = W.ToggleButtons(options=["konzentrisch", "zigzag", "explizit"], value=s["infill"],
                             style={"button_width": "96px"})
    density = W.IntSlider(value=s["density"], min=0, max=100, step=5, description="Dichte (%)",
                          style=LABEL, layout=FULL)
    walls = W.IntSlider(value=s["walls"], min=0, max=5, description="Wände",
                        style=LABEL, layout=FULL)
    printer = W.Dropdown(options=list(lsi_lite.PRINTERS), value=s["printer"],
                         description="Drucker", style=LABEL, layout=FULL)
    material = W.ToggleButtons(options=list(lsi_lite.MATERIALS), value=s["material"],
                               style={"button_width": "70px"})
    run = W.Button(description="Slicen", button_style="primary", icon="play",
                   layout=W.Layout(width="180px", height="38px"))
    status = W.HTML()
    progress = W.HTML()
    log, preview, view, result = W.Output(), W.Output(), W.Output(), W.Output()
    tabs = W.Tab(children=[log, preview, view], layout=W.Layout(display="none"))
    for i, name in enumerate(("Protokoll", "Schnitt", "3D-Ansicht")):
        tabs.set_title(i, name)

    def on_source(change):
        upload.layout.display = "" if change["new"] == UPLOAD else "none"

    def on_nozzle(change):
        window.value = f'<div class="lsi-hint">{window_text(change["new"])}</div>'

    def input_file():
        """Pfad der gewaehlten .vtu; None mit Hinweis in status, wenn keine da ist."""
        if source.value != UPLOAD:
            return lsi_lite.HERE / lsi_lite.EXAMPLES[source.value.split(": ")[1]][0]
        files = [(n, c) for n, c in uploaded_files(upload) if n.lower().endswith(".vtu")]
        if not files:
            status.value = "<b>Erst eine .vtu-Datei wählen.</b>"
            return None
        target = lsi_lite.HERE / "eingabe" / Path(files[0][0]).name
        target.parent.mkdir(exist_ok=True)
        target.write_bytes(files[0][1])
        return target

    def on_run(_):
        vtu = input_file()
        if vtu is None:
            return
        run.disabled, status.value = True, ""
        tabs.layout.display, tabs.selected_index = "", 0
        for out in (log, preview, view, result):
            out.clear_output()
        try:
            with log:
                gcode, png, paths, strokes = lsi_lite.slice_vtu(
                    vtu, show=lambda html: setattr(progress, "value", html),
                    nozzle=nozzle.value, max_tilt=tilt.value, infill=infill.value,
                    density=density.value, walls=walls.value, printer=printer.value,
                    material=material.value, build_dir=build_dir.value)
        except lsi_lite.InputError as err:
            status.value = f"<b>Abbruch:</b> {err}"
            return
        finally:
            run.disabled = False
        with preview:
            display(W.Image(value=png.read_bytes(), format="png", layout=W.Layout(max_width="100%")))
        with view:
            lsi_lite.show_paths(paths, strokes=strokes).show()
        with result:
            if IN_COLAB:
                from google.colab import files
                files.download(str(gcode))
                print(f"G-Code wird heruntergeladen: {gcode.name}")
            else:
                print(f"G-Code: {gcode}")
        tabs.selected_index = 2

    source.observe(on_source, names="value")
    nozzle.observe(on_nozzle, names="value")
    run.on_click(on_run)
    cards = W.Box([
        card("Bauteil", source, upload, build_dir,
             hint("Baurichtung: welche Achse des FEM-Modells nach oben zeigt; "
                  "<i>auto</i> = bei Beispielen deren Richtung, sonst +z.")),
        card("Schichten", nozzle, window, tilt,
             hint("Wie man die Neigung für den eigenen Drucker bestimmt: README, "
                  "Abschnitt <i>Maximale Neigung ermitteln</i>.")),
        card("Füllung", infill, density, walls,
             hint("<i>explizit</i>: Bahnen entlang der Zugrichtung, mit Lücken.")),
        card("Drucker", printer, material),
    ], layout=W.Layout(display="flex", flex_flow="row wrap", width="100%"))
    title = W.HTML(CSS + '<div class="lsi-title"><h2>LSI Lite</h2><p>Lastoptimiert slicen aus '
                   'einem FEM-Ergebnis (.vtu). Den G-Code vor dem Druck immer ansehen.</p></div>')
    display(W.VBox([title, cards, W.HBox([run, status], layout=W.Layout(align_items="center", grid_gap="14px")),
                    progress, result, tabs], layout=W.Layout(max_width="1100px")))


app()